## 문제 정의

- 뉴스그룹 본문을 입력받아 20개 카테고리 중 하나를 예측한다.
- 검증 지표는 정확도와 클래스별 precision, recall, F1이다.

## 시드 고정

시드(seed)는 난수의 시작값이야. Python, NumPy, PyTorch에 42를 설정해 데이터 섞기와 모델 초기화를 재현하기 쉽게 만든다. CUDA는 NVIDIA GPU를 사용할 때 선택된다.

In [ ]:
# 필요한 패키지: numpy pandas matplotlib scikit-learn nltk wordcloud gensim torch tqdm
import copy
import hashlib
import os
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from wordcloud import WordCloud

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 데이터 수집

20 Newsgroups는 20개 카테고리에 속한 게시글 18,846건을 담고 있다. 헤더·푸터·인용문을 제거해 본문을 가져오고, 깊은 복사(`deepcopy`)로 원본과 작업용 데이터를 분리한다.

In [ ]:
# https://scikit-learn.org/stable/datasets/real_world.html#the-20-newsgroups-text-dataset
original_data = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"))
data = copy.deepcopy(original_data)

In [ ]:
# 데이터 설치 이후, 타입 확인
print(type(data))

## 하기 내용은 스스로 공부를 위해서 상기 타입이 나오기 전에 기준서처럼 작성하는 셀입니다.

### 표(DataFrame)라면

행·열 크기와 열 이름을 확인하고 첫 몇 줄을 본다. 표 형태의 데이터를 만났을 때 이 셀의 주석을 풀어 실행한다.

In [ ]:
# print("크기(행, 열):", data.shape)
# print("열 이름들:", data.columns)
# print(data.head())

### Bunch 형태라면

Bunch는 이름으로 항목을 꺼낼 수 있는 묶음이다. 현재 20 Newsgroups는 이 형태라서 키 목록을 확인한 뒤 `data`(문서), `target`(정답 번호), `target_names`(카테고리 이름), `DESCR`(데이터 설명)을 확인한다.

In [ ]:
print("들어있는 항목들:", data.keys())

> 인사이트 : 'data' : 데이터, 'filenames' : 파일명, 'target_names' : 대상_이름, 'target' : 대상_설명, 'DESCR' : 설명

In [ ]:
# 글 본문 확인하기
texts = data.data
print("몇개의 글이 있나요?", len(texts)) # print 한 값에 2개를 표현할때 , 를 사용하여 표현해야 오류가 안남
print("첫 번째 글의 자료형은?", type(texts[0]))

In [ ]:
labels = data.target
print("몇개의 정답이 있나요?", labels.shape)
print("첫 다섯 글의 정답 번호", labels[:5])

In [ ]:
# 숫자로 표기되는 정답번호라 보기가 불편하니 이름으로 확인하자
target_names = data.target_names
print(pd.Series(target_names)) # 시드 고정에서 pandas를 불러왔으니 Series로 순서 숫자와 dtype까지 한번에 확인하는 명령어 사용

> 인사이트 : 한국어 번역<br>
0 대안적 무신론<br>
1 컴퓨터 그래픽<br>
2 컴퓨터 운영 체제(MS-Windows) 기타<br>
3 컴퓨터 시스템(IBM) PC 하드웨어<br>
4 컴퓨터 시스템(Mac) 하드웨어<br>
5 컴퓨터 윈도우 X<br>
6 기타 판매<br>
7 자동차 관련 정보<br>
8 오토바이 관련 정보<br>
9 야구 관련 정보<br>
10 하키 관련 정보<br>
11 암호 과학<br>
12 전자 과학<br>
13 의학 과학<br>
14 우주 과학<br>
15 기독교 사회 종교<br>
16 총기 정치 토론<br>
17 중동 정치 토론<br>
18 기타 정치 토론<br>
19 기타 종교 토론<br>

일부 나누는 기준이 있어 보이니 사이킷 런 홈페이지에서 확인해본다.

alt, comp, misc, rec, sci, soc, talk 라벨 앞에 해당 목록이 보인다. 해당 라벨을 가진 글들을 하나씩 확인해서 어떤 정보를 저렇게 표현했는지 확인한다.

In [ ]:
print(data.DESCR[:500])

### 리스트 또는 배열이라면

전체 개수와 첫 번째 데이터, 첫 번째 데이터의 자료형을 확인한다. 리스트나 배열을 만났을 때 이 셀의 주석을 풀어 실행한다.

In [ ]:
# print("전체 개수:", len(data))
# print("첫 번째 데이터:", data[0])
# print("첫 번째 데이터의 타입:", type(data[0]))

## 데이터 분할

홀드아웃 방식으로 train 60%, val 20%, test 20%를 만든다. train은 학습, val은 학습 중 성능 확인과 모델 선택, test는 최종 평가에 사용한다.

`train_test_split`은 한 번에 둘로만 나눈다. 먼저 전체의 20%를 test로 떼고, 남은 80%의 25%를 val로 떼면 전체의 20%가 된다. 남은 60%가 train이다. 두 번 모두 `stratify`에 해당 단계의 라벨을 넣어 카테고리 비율을 유지하고, `random_state=42`로 같은 분할을 재현한다.

In [ ]:
train_val_inputs, test_inputs, train_val_targets, test_targets = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)

In [ ]:
train_inputs, val_inputs, train_targets, val_targets = train_test_split(
    train_val_inputs, train_val_targets, test_size=0.25,
    random_state=SEED, stratify=train_val_targets
)
print({name: (len(split), round(len(split) / len(texts), 3))
       for name, split in zip(("train", "val", "test"), (train_inputs, val_inputs, test_inputs))})

## 정제 전 EDA
### 클래스 분포

정제 전 EDA 결과를 기록해 어떤 실험이 필요한지 판단할 때 사용한다.

EDA(탐색적 데이터 분석)는 train 데이터로 문서의 특징을 먼저 살펴보는 단계다. 분할 간 중복 확인에서만 val과 test의 해시를 함께 사용한다.

In [ ]:
# 카테고리별로 문서 수가 균형 잡혀있는지 확인
class_counts = pd.Series(train_targets).value_counts().reindex(range(len(target_names)), fill_value=0)
class_counts.index = target_names
display(class_counts.to_frame("문서 수"))

In [ ]:
# bar plot으로 labels를 출력하고 클래스 이름으로 표시
class_counts.plot.bar(figsize=(12, 4), ylabel='Documents')
plt.tight_layout()
plt.show()

> **인사이트 :** alt.atheism과 talk.religion.misc, talk.politics.misc 3개의 클래스가 다른 클래스에 비해 부족해보인다. 따라서 분할 방시에 있어서는 비율에 대해서도 고려될 수 있도록 수정한다.

### 문서 길이 분포

NLTK의 `word_tokenize`는 문서를 단어와 문장부호 등 토큰으로 나눈다. `punkt`와 `punkt_tab`은 토큰화용 자료이며, `stopwords`는 정제 단계에서 쓸 불용어 목록이다. 히스토그램은 토큰 개수 범위마다 문서가 몇 건인지 보여준다.

상기 내용이 너무 어려워 쉽게 말하자면 정제 전 EDA를 위해 train 문서마다 NLTK의 word_tokenize()를 적용했다. 공백으로 구분된 단어와 쉼표·느낌표 같은 문장부호가 토큰으로 나뉘며, 문서별 토큰 개수를 세어 길이 분포를 확인했다.

In [ ]:
# 문서 길이 분포는 베이스라인의 히스토그램을 사용
for resource in ("punkt", "punkt_tab", "stopwords"):
    nltk.download(resource, quiet=True)

raw_tokens = [word_tokenize(text) for text in train_inputs]

def show_lengths(tokens, title):
    lengths = [len(doc) for doc in tokens]
    plt.hist(lengths, bins=30, alpha=0.7, label="Train Data")
    plt.xlabel("Message Length (words)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.title(title)
    plt.show()
    display(pd.Series(lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
    return lengths

raw_lengths = show_lengths(raw_tokens, "Distribution of Message Lengths (raw)")

> **인사이트 :** 토큰 길이의 중앙값은 101개로 대부분의 메시지는 짧지만, 최댓값이 약 40,000에 달하는 극단적 이상치가 존재한다. 이를 기준으로 max_len을 정하면 padding 낭비가 크므로, 전체의 95%를 커버하는 705 토큰을 max_len 값을 정제 후, 변화와 비교한다.

### 단어 빈도

`Counter`는 토큰별 등장 횟수를 센다. 빈도 상위 목록은 어떤 단어와 기호가 반복되는지 확인하는 데 쓴다.

In [ ]:
# 제일 많이 나오는 단어 확인
def word_counts(tokens):
    return Counter(word for doc in tokens for word in doc)

raw_counts = word_counts(raw_tokens)
display(raw_counts.most_common(20))

워드클라우드는 자주 나온 토큰을 크게 표시한다. 정확한 횟수는 빈도 목록에서 확인한다.

In [ ]:
# 워드 클라우드로 시각화
def show_cloud(counts):
    cloud = WordCloud(width=900, height=360, background_color="white", random_state=SEED)
    plt.figure(figsize=(10, 4))
    plt.imshow(cloud.generate_from_frequencies(counts), interpolation="bilinear")
    plt.axis("off")
    plt.show()

show_cloud(raw_counts)

> **인사이트 :** 정제 전 워드클라우드에서 the, to, and 같은 흔한 단어와 >, *, # 같은 기호가 크게 나타났다. 기본 정제를 적용한 뒤 어떤 토큰이 빈도 상위에 남는지 비교한다. 기본 불용어 정제 시, 특수문자는 전부 제거하는 방향을 확인한다.

희귀 단어 비율의 분모는 전체 등장 횟수가 아니라 고유 토큰 수다. 여기서 희귀 단어는 train에서 딱 한 번 나온 토큰이다.

In [ ]:
# 1번만 나오는 희귀 단어 확인 및 비율
def singleton_words(counts):
    rare = {word for word, count in counts.items() if count == 1}
    print(f"한 번 나온 단어: {len(rare):,}개 ({len(rare) / max(len(counts), 1):.2%} of vocabulary)")
    return rare

raw_rare = singleton_words(raw_counts)

> **인사이트 :** 정제 전 train에서 한 번만 등장한 토큰은 90,083종으로, 전체 고유 토큰의 60.71%였다. 원문 예시에서 어떤 토큰이 한 번만 나왔는지 확인하고, 정제 후 개수와 비율을 비교한다.

In [ ]:
# 예시를 print하여 확인하고 문서 리스트로 뽑아두기
rare_documents = [i for i, doc in enumerate(raw_tokens) if raw_rare.intersection(doc)]
for i in rare_documents[:3]:
    print(i, sorted(raw_rare.intersection(raw_tokens[i]))[:8])
    print(train_inputs[i][:300])

### 어휘 크기

In [ ]:
# 고유 단어 수가 몇 개인지 확인하기
print('정제 전 고유 토큰 수', len(raw_counts))

### 노이즈 확인

해시(hash)는 문서를 비교할 때 쓰는 고정 길이 지문이다. 이 셀에서는 앞뒤 공백을 제거한 뒤 SHA-256 지문을 만들어 같은 내용이 몇 번 반복됐는지 센다.

In [ ]:
# 빈 문서, 중복 문서를 hash를 사용해서 확인하기
'''
return 으로 계속 돌려줄껀데, 
SHA-256 기능을 쓸 수 있게 해주는 파이썬 기본 모듈
sha256은 글을 받아서 일정 한 길이의 '디지털 지문'을 만드는 방법
text.strip() 본문 앞뒤의 공백과 줄바꿈을 제거한다.
encode("utf-8") 글자를 컴퓨터가 처리할 수 있는 숫자(바이트)로 바꾸는 것
hexdigest() 만들어진 지문을 a3f2같은 문자열로 꺼내줌
'''
def document_hash(text):
    return hashlib.sha256(text.strip().encode("utf-8")).hexdigest()

train_hashes = Counter(map(document_hash, train_inputs)) # map()은 train 글마다 document_hash()를 적용한다.
empty_documents = [i for i, text in enumerate(train_inputs) if not text.strip()]
duplicate_documents = [i for i, text in enumerate(train_inputs)
                       if train_hashes[document_hash(text)] > 1]
print("빈 문서", len(empty_documents))
print("중복 건수(첫 등장 제외, 빈 문서 포함)", sum(n - 1 for n in train_hashes.values()))

> **인사이트 :**
Train 데이터에서 본문이 비어 있거나 공백만 있는 문서가 306건 확인됐다. 헤더·푸터·인용문을 제거하는 과정에서 일부 문서의 본문이 비었을 가능성이 있다. 빈 문서는 본문을 바탕으로 카테고리를 학습하는 데 단서를 주지 못하므로, 제거 전 원문과 비교해 비게 된 원인을 확인한 뒤 학습 데이터에서 제외한다.

분할 간 중복은 같은 본문이 train과 val 또는 test에 걸쳐 있는지 확인하는 점검이다. 세 분할 쌍에서 같은 해시를 가진 비어 있지 않은 고유 문서 수를 센다.

In [ ]:
# train/val/test 간 중복도 확인
# 분할 간 누수 점검은 비어 있지 않은 문서의 해시 집합만 사용한다.
hash_sets = {
    name: {document_hash(text) for text in split if text.strip()}
    for name, split in (("train", train_inputs), ("val", val_inputs), ("test", test_inputs))
}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    print(f"{a}와 {b}에 똑같이 들어 있는 본문: {len(hash_sets[a] & hash_sets[b])} 종류")

> **인사이트 :** 라벨 값도 제외가 되야되기 때문에 해당 추가로 짝지어 비교한다.

In [ ]:
print("val 빈 문서:", sum(not text.strip() for text in val_inputs))
print("test 빈 문서:", sum(not text.strip() for text in test_inputs))

| 데이터 분할 | 빈 문서 수 |
|---|---:|
| train | 306 |
| val | 95 |
| test | 114 |
| **합계** | **515** |

<br>

> **인사이트 :** remove=("headers", "footers", "quotes")를 적용한 뒤의 데이터에서 본문이 비어 있거나 공백, 줄바꿈만 남은 문서는 총 515개이다. 

정규표현식(regex)은 글자 패턴을 찾는 규칙이다. 특수문자·숫자·이메일 주소를 각각 찾아 전체 등장 횟수와 포함된 문서 수를 센다.

In [ ]:
# 특수문자, 숫자, 이메일 주소가 얼마나 섞였는지 보기
noise_patterns = {
    "특수문자": re.compile(r"[^\w\s]|_"),
    "숫자": re.compile(r"\d"),
    "이메일": re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"),
}
noise_documents = {}
for name, pattern in noise_patterns.items():
    counts = [len(pattern.findall(text)) for text in train_inputs]
    noise_documents[name] = [i for i, count in enumerate(counts) if count]
    print(name, "등장 수", sum(counts), "포함 문서 수", len(noise_documents[name]))

노이즈 종류별로 해당 문서의 train 위치를 리스트에 저장해 뒀다. 그중 앞의 두 문서씩 일부를 출력해 실제로 어떤 형태가 섞여 있는지 확인한다.

In [ ]:
# 예시를 print하여 확인하고 문서 리스트로 뽑아두기
for name, indices in noise_documents.items():
    for i in indices[:2]:
        print(name, i, train_inputs[i][:300])

> **인사이트 :** 헤더·푸터·인용문을 제거한 뒤 빈 문서와 분할 간에 본문이 같은 문서를 확인했다. 추가 정제에서는 본문에 남아 있는 이메일 주소와 특수문자를 제거한다. 숫자는 제품 사양, 버전, 금액처럼 카테고리를 판단하는 데 도움이 되는 정보를 담을 수 있으므로 유지한다.

### n-gram 분석

n-gram은 연달아 등장한 토큰 묶음이다. 여기서는 두 토큰짜리 bigram을 세고, 문서 경계를 넘어 단어쌍을 만들지 않는다.

In [ ]:
# 자주 같이 나오는 단어쌍 확인
def top_bigrams(tokens):
    return Counter(pair for doc in tokens for pair in zip(doc, doc[1:])).most_common(20)

display(top_bigrams(raw_tokens))

> **인사이트 :** 불용어 제거 전 조합

[(('>', "'"), 49913),
 (('--', '--'), 48522),
 (("'", 'AX'), 45738),
 (('AX', '>'), 45557),
 (('*', '*'), 11685),
 (('of', 'the'), 11142),
 ((',', 'and'), 9700),
 (('.', 'I'), 8136),
 (('in', 'the'), 7273),
 (('.', 'The'), 7112),
 ((',', 'but'), 5334),
 ((',', 'I'), 4565),
 ((')', '.'), 4506),
 (('to', 'the'), 4430),
 ((',', 'the'), 4112),
 (('on', 'the'), 3924),
 (('do', "n't"), 3582),
 (('to', 'be'), 3381),
 (('.', 'It'), 3337),
 (("'", 'MAX'), 3275)]

### 데이터 확인 이후, 추가 EDA
- alt, comp, misc, rec, sci, soc, talk 라벨 앞에 해당 목록이 보인다. 해당 라벨을 가진 글들을 하나씩 확인해서 어떤 정보를 저렇게 표현했는지 확인한다.<br>

In [ ]:
prefix = "rec"  # alt, comp, misc, rec, sci, soc, talk 중 하나
examples_per_label = 2

print(f"===== {prefix} 그룹 =====")

for label_id, label_name in enumerate(target_names):
    if label_name.split(".")[0] != prefix:
        continue

    documents = [
        text.strip()
        for text, label in zip(train_inputs, train_targets)
        if label == label_id and text.strip()
    ]

    print(f"\n[{label_name}] train 글 {len(documents)}편")
    for i, text in enumerate(documents[:examples_per_label], start=1):
        print(f"\n예시 {i}\n{text[:600]}")

> 인사이트 : 카테고리 라벨 앞부분은 대분류이고, 뒤에는 소분류 이다.

| 라벨 앞부분 | 영어 원형 | 한국어로 적으면 |
|---|---|---|
| `alt` | **alternative** | 자유 주제 |
| `comp` | **computer** | 컴퓨터 |
| `misc` | **miscellaneous** | 기타 |
| `rec` | **recreation** | 여가·취미 |
| `sci` | **science** | 과학 |
| `soc` | **social** | 사회·문화 |
| `talk` | **talk** | 토론 |

- 공백만 있는 문서와 원본 문서를 1x2로 출력하여 비교한다. 해당 문서가 학습하는데 카테고리 단서를 주는지 못주는지를 확인하고 제외할지 아닐지를 결정한다.

In [ ]:
from html import escape
from IPython.display import HTML, display

# 헤더·푸터·인용문을 제거하기 전 원문을 불러온다.
unfiltered_data = fetch_20newsgroups(subset="all", remove=())

# 두 데이터에서 같은 위치가 같은 문서인지 확인한다.
assert np.array_equal(unfiltered_data.filenames, data.filenames)

# 기존과 같은 조건으로 train 문서가 전체 데이터의 몇 번에 있었는지 찾는다.
train_val_ids, _ = train_test_split(
    np.arange(len(texts)),
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)
train_ids, _ = train_test_split(
    train_val_ids,
    test_size=0.25,
    random_state=SEED,
    stratify=labels[train_val_ids],
)

# 빈 문서 목록에서 앞의 3개를 비교한다.
for train_pos in empty_documents[:3]:
    original_id = train_ids[train_pos]
    original_text = unfiltered_data.data[original_id]
    current_text = train_inputs[train_pos]
    category = target_names[train_targets[train_pos]]

    print(f"train 위치 {train_pos} | 정답 카테고리 {category}")

    display(HTML(f"""
    <table style="width: 100%; table-layout: fixed;">
      <tr><th>제거 전 원문</th><th>제거 후 본문</th></tr>
      <tr>
        <td style="vertical-align: top;">
          <pre style="white-space: pre-wrap;">{escape(original_text)}</pre>
        </td>
        <td style="vertical-align: top;">
          <pre style="white-space: pre-wrap;">{escape(repr(current_text))}</pre>
        </td>
      </tr>
    </table>
    """))

> **인사이트 :**
Train에서 본문이 비어 있는 문서 306건을 확인했다. 그중 일부를 제거 전 원문과 비교한 결과, 확인한 예시에서는 카테고리를 추론할 만한 내용이 거의 없었다. 모델에 입력할 본문이 비어 있으므로 해당 문서를 학습 데이터에서 제외한다.

| EDA 항목 | 확인한 내용 | 정제에 반영할 점 |
|---|---|---|
| 문서 길이 | 정제 전 train 문서의 토큰 수 분포를 확인 | 정제 후 train의 길이 분포를 다시 보고 `max_len`을 확인 |
| 워드클라우드 | `the`, `to`, `and` 같은 흔한 단어와 `>`, `*`, `#` 같은 기호 사용진도가 높음 | 기본 영어 불용어와 특수문자를 제거하고, 정제 후 빈도 상위에 무엇이 남는지 비교 |
| 한 번 나온 토큰 | **90,083종**, 전체 고유 토큰의 **60.71%** | 정제 후 개수와 비율을 다시 비교 |
| 빈 문서 | 헤더·푸터·인용문 제거 후 **train 306건, val 95건, test 114건**이 빈 문서가 됨, 확인한 원문 예시에서도 카테고리 단서가 거의 없음 | 세 분할에 같은 기준을 적용해 **빈 본문과 짝이 되는 라벨을 함께 제외**하여 정제하면서 새로 빈 문서가 생기는지도 확인 |
| 분할 간 같은 본문 | `train–val` 18종류, `train–test` 11종류, `val–test` 7종류에서 같은 본문 중복 | 빈 문서와 별개로 같은 글을 학습과 평가에서 다시 보는 일을 피하도록 **분할 간 중복을 따로 처리할 항목**으로 처리 필요 |
| 이메일·특수문자·숫자 | 헤더·푸터·인용문을 뺀 본문에도 남아 있다는 것을 확인, 숫자는 버전, 제품 사양, 금액의 단서가 될 수 있다고 판단, 숫자 외 제거 실시 | **이메일 주소를 통째로 먼저 제거**하고, 특수문자를 공백으로 바꿔. 영문자와 숫자는 보존 |
| 연속 토큰 쌍 | 정제 전 train에서 어떤 토큰이 자주 붙어 나오는지 확인하는 단계 | 정제 후에도 같은 집계를 해서 기호나 흔한 단어 조합이 줄었는지 비교필요 |

따라서 소문자 변환, 이메일 추가 제거, 특수문자 제거, 빈문서 제거가 필요

## 정제 클리닝

불용어는 정제 규칙에서 제거할 단어다. NLTK 영어 목록을 기본으로 쓰고, 추가 목록은 train EDA를 읽은 뒤 채운다.

In [ ]:
# train EDA 검토 후 필요한 단어만 여기에 추가한다.
extra_stopwords = set()
stop_words = set(stopwords.words("english")) | extra_stopwords

`clean_text`는 소문자로 바꾸고 영문자 이외의 문자를 공백으로 바꾼 뒤 불용어를 걸러낸다. 세 분할 모두 같은 함수를 거친다.

In [ ]:
# 이메일 주소를 찾는 "규칙(패턴)"을 미리 만들어둔다.
# re.compile: 규칙을 한 번만 만들어두고 여러 번 재사용 (속도와 가독성에 유리)
email_pattern = re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}")

In [ ]:
def clean_text(text):
    # 1) 이메일을 통째로 지운다. (공백으로 치환)
    #    반드시 특수문자 제거보다 먼저! 그래야 @ 와 . 이 남아 있는 상태에서 이메일을 찾을 수 있다.
    text = email_pattern.sub(" ", text)

    # 2) 소문자로 바꾼 뒤, 영문 소문자 / 숫자 / 공백이 "아닌" 문자는 모두 공백으로 바꾼다.
    #    [^...] 안의 ^ 는 "~가 아닌 것"이라는 뜻
    text = re.sub(r"[^a-z0-9\s]", " ", text.lower())

    # 3) 연속된 공백과 줄바꿈을 공백 하나로 정리한다.
    #    split(): 공백이 여러 개든 줄바꿈이든 한꺼번에 기준으로 잘라 단어 리스트를 만든다.
    #    " ".join(...): 그 단어들을 공백 하나로 다시 이어 붙인다.
    return " ".join(text.split())

In [ ]:
sample = "Contact: John.Doe@Abc.com   Price: $100!!\n\nHello, World."
print(clean_text(sample))     # contact price 100 hello world
print(repr(clean_text("!!! ??? ...")))   # '' → 정제 후 빈 문서가 되는 예

In [ ]:
def clean_split(inputs, targets, split_name):
    clean_inputs = []     # 정제된 문서를 담을 리스트
    clean_targets = []    # 그 문서의 정답을 담을 리스트 (항상 같은 길이로 유지!)
    already_empty = 0     # 정제 전부터 비어 있던 문서 수
    newly_empty = 0       # 정제하고 나니 비게 된 문서 수

    # zip: 문서와 정답을 한 쌍씩 묶어서 같이 꺼낸다.
    for text, label in zip(inputs, targets):
        cleaned = clean_text(text)

        if cleaned:
            # 정제 후에도 내용이 남아 있으면 → 문서와 정답을 "같이" 저장
            clean_inputs.append(cleaned)
            clean_targets.append(label)
        elif text.strip():
            # 원본엔 내용이 있었는데 정제 후 사라짐 (예: 기호만 있던 문서)
            newly_empty += 1
        else:
            # 원본부터 비어 있던 문서 (remove 옵션으로 본문이 사라진 경우 등)
            already_empty += 1

    print(
        f"{split_name} | 기존 빈 문서 {already_empty}건 | "
        f"정제 후 새로 빈 문서 {newly_empty}건 | 남은 문서 {len(clean_inputs)}건"
    )
    return clean_inputs, clean_targets

In [ ]:
train_clean, train_clean_targets = clean_split(train_inputs, train_targets, "train")
val_clean,   val_clean_targets   = clean_split(val_inputs,   val_targets,   "val")
test_clean,  test_clean_targets  = clean_split(test_inputs,  test_targets,  "test")

assert len(train_clean) == len(train_clean_targets)
assert len(val_clean)   == len(val_clean_targets)
assert len(test_clean)  == len(test_clean_targets)
print(len(train_clean), len(val_clean), len(test_clean))

## 단어 토큰화

문서 하나를 토큰 리스트 하나로 바꾼다. 이렇게 만든 세 분할의 토큰을 각 임베딩에서 공통으로 사용한다.

In [ ]:
train_sentences = [word_tokenize(text) for text in train_clean]
test_sentences = [word_tokenize(text) for text in test_clean]

In [ ]:
val_sentences = [word_tokenize(text) for text in val_clean]

In [ ]:
assert len(train_sentences) == len(train_clean_targets)
assert len(val_sentences)   == len(val_clean_targets)
assert len(test_sentences)  == len(test_clean_targets)

## 정제 후 EDA

### 클래스 분포

In [ ]:
# 정제 전/후 클래스별 문서 수 (train 기준, 클래스 번호 순서로 정렬)
before = pd.Series(train_targets).value_counts().sort_index()
after = pd.Series(train_clean_targets).value_counts().sort_index()

# 한 표로 합치고, 클래스 번호를 카테고리 이름으로 바꾼다.
compare = pd.DataFrame({"정제 전": before, "정제 후": after})
compare["제거"] = compare["정제 전"] - compare["정제 후"]
compare.index = data.target_names

display(compare)
compare["정제 후"].plot(kind="bar")

### 문서 길이 분포

In [ ]:
# 문서마다 토큰이 몇 개인지 센다.
clean_lengths = [len(tokens) for tokens in train_sentences]

plt.hist(clean_lengths, bins=30)
plt.title("Distribution of Message Lengths (clean)")
plt.xlabel("Tokens per document")
plt.ylabel("Number of documents")
plt.show()

# 중앙값과 90·95·99분위수를 확인한다.
display(pd.Series(clean_lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

95분위수는 짧은 쪽부터 정렬했을 때 약 95%가 그 이하에 들어오는 길이다. 이 값과 베이스라인의 280을 각각 적용할 때 잘리는 train 문서 비율을 비교한다.

In [ ]:
# max_len을 train 분포로 정할 수 있도록 확인
candidate_max_len = max(1, int(np.ceil(np.percentile(clean_lengths, 95))))
for length in (304, candidate_max_len): # 숫자를 바꿔줄 수 있음 
    print(length, f"토큰이 잘리는 train 문서 비율 {np.mean(np.array(clean_lengths) > length):.2%}")

> **인사이트 :** 정제 후 train 길이 분포와 잘리는 문서 비율이 95%가 607.5로 max_len을 608로 설정한다.

### 단어 빈도

정제 전과 같은 방식으로 빈도, 워드클라우드, 한 번 나온 토큰을 다시 확인한다.

In [ ]:
# 제일 많이 나오는 단어, 워드 클라우드, 1번만 나오는 희귀 단어 및 비율 확인
# 단어별로 몇 번 등장했는지 센다.
clean_counts = Counter(
    word for sentence in train_sentences for word in sentence
)

# 제일 많이 나오는 단어 20개를 확인한다.
display(clean_counts.most_common(20))

# 단어 빈도를 워드클라우드로 시각화한다.
# 앞에서 센 단어를 그대로 표시하도록 WordCloud의 자체 불용어는 비운다.
if clean_counts:
    cloud = WordCloud(
        width=1000,
        height=500,
        background_color="white",
        stopwords=set(),
        random_state=42,
    ).generate_from_frequencies(clean_counts)

    plt.figure(figsize=(12, 6))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()

# 전체 train에서 딱 한 번만 나온 단어를 모은다.
clean_rare = [
    word for word, count in clean_counts.items() if count == 1
]

# 고유 단어 중 한 번만 등장한 단어의 비율을 확인한다.
rare_ratio = len(clean_rare) / len(clean_counts) if clean_counts else 0

print("고유 단어 수:", len(clean_counts))
print(f"한 번 나온 단어: {len(clean_rare):,}개 ({rare_ratio:.2%})")
print("한 번 나온 단어 예시:", clean_rare[:20])

> **인사이트 :**한 번 나온 단어 예시로는 복수와 단수를 나누는 s나 일부 오타 단어가 있는걸로 보여 복수 -> 단수로 전부 바꾸고 오타는 일부를 확인 후에 불용어로 삭제하는 방안을 고민한다.

### 어휘 크기

In [ ]:
# 고유 단어 수가 몇 개인지 확인하기
# set은 같은 단어를 하나만 남긴다.
clean_vocabulary = {
    word for sentence in train_sentences for word in sentence
}

print("고유 단어 수:", len(clean_vocabulary))

### 노이즈 확인

정제 후 토큰이 하나도 남지 않은 문서의 위치를 저장한다. 문서 해시와 정규표현식을 다시 적용해 중복과 노이즈가 얼마나 남았는지 확인한다.

In [ ]:
# 정제 후 토큰들을 공백으로 연결해 확인용 문서로 만든다.
clean_documents = [" ".join(sentence) for sentence in train_sentences]

# 빈 문서 확인하기
clean_empty_documents = [
    i for i, text in enumerate(clean_documents) if not text.strip()
]
print("빈 문서:", len(clean_empty_documents))

# 중복 문서를 hash로 확인하기
# 같은 내용은 같은 해시값으로 바뀐다.
clean_hashes = [
    hashlib.sha256(text.strip().encode("utf-8")).hexdigest()
    for text in clean_documents
]
clean_hash_counts = Counter(clean_hashes)

print(
    "중복 건수(첫 등장 제외, 빈 문서 포함):",
    sum(count - 1 for count in clean_hash_counts.values())
)

# 중복된 문서의 위치를 저장한다. 첫 등장 문서도 포함한다.
clean_duplicate_documents = [
    i for i, value in enumerate(clean_hashes)
    if clean_hash_counts[value] > 1
]

# 특수문자, 숫자, 이메일 주소가 얼마나 남았는지 보기
clean_noise_patterns = {
    "특수문자": re.compile(r"[^\w\s]|_"),
    "숫자": re.compile(r"\d"),
    "이메일": re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"),
}

clean_noise_documents = {}

for name, pattern in clean_noise_patterns.items():
    counts = [len(pattern.findall(text)) for text in clean_documents]

    # 해당 내용이 포함된 문서의 위치를 저장한다.
    clean_noise_documents[name] = [
        i for i, count in enumerate(counts) if count > 0
    ]

    print(
        name,
        "등장 수:", sum(counts),
        "포함 문서 수:", len(clean_noise_documents[name])
    )

    # 예시 문서 2개를 각각 앞 300글자까지 확인한다.
    for i in clean_noise_documents[name][:2]:
        print(f"문서 위치 {i}:")
        print(clean_documents[i][:300])
        print()

### n-gram 분석

In [ ]:
# 자주 같이 나오는 단어쌍 확인하기
clean_bigram_counts = Counter(
    (sentence[i], sentence[i + 1])
    for sentence in train_sentences
    for i in range(len(sentence) - 1)
)

# 가장 많이 등장한 단어쌍 20개를 표로 확인한다.
clean_bigram_table = pd.DataFrame(
    [
        (first, second, count)
        for (first, second), count in clean_bigram_counts.most_common(20)
    ],
    columns=["첫 번째 토큰", "두 번째 토큰", "등장 횟수"]
)

display(clean_bigram_table)

### 전처리 전후 비교

같은 train 문서의 정제 전후를 나란히 확인한다. 여러 조건으로 선택된 문서가 겹치면 `dict.fromkeys`로 문서 위치를 한 번만 남긴다.

In [ ]:
# 전처리 전후 비교
# train_inputs: 정제 전 학습 문서
# clean_text: 앞에서 정의해서 사용한 정제 함수

comparison_rows = []

for i, before in enumerate(train_inputs):
    after = clean_text(before)

    # 정제로 내용이 바뀐 문서만 모은다.
    if before != after:
        comparison_rows.append({
            "정제 전 문서 위치": i,
            "정제 전": before,
            "정제 후": after,
            "정제 후 빈 문서 여부": not after.strip(),
        })

    # 예시 5개를 모으면 멈춘다.
    if len(comparison_rows) == 5:
        break

# 긴 글도 확인할 수 있도록 각 칸에 최대 1,000글자를 표시한다.
with pd.option_context("display.max_colwidth", 1000):
    display(pd.DataFrame(comparison_rows))

In [ ]:
# train, val, test 각각 정제 전후의 빈 문서 수를 비교한다.
empty_comparison = []

for name, documents in [
    ("train", train_inputs),
    ("val", val_inputs),
    ("test", test_inputs),
]:
    # 정제 전: 원래 비어 있거나 공백만 있는 문서
    before_count = sum(not text.strip() for text in documents)

    # 정제 후: 같은 정제 함수를 적용했을 때 비게 되는 문서
    after_count = sum(
        not clean_text(text).strip() for text in documents
    )

    empty_comparison.append({
        "데이터": name,
        "전체 문서 수": len(documents),
        "정제 전 빈 문서": before_count,
        "정제 후 빈 문서": after_count,
        "정제로 추가된 빈 문서": after_count - before_count,
    })

display(pd.DataFrame(empty_comparison))

In [ ]:
def check_pairs(name, inputs, targets, clean, clean_targets, sentences):
    print(f"===== {name} =====")

    # 1) 변수별 개수를 출력한다.
    sizes = {
        "원본 문서(inputs)": len(inputs),
        "원본 정답(targets)": len(targets),
        "정제 문서(clean)": len(clean),
        "정제 정답(clean_targets)": len(clean_targets),
        "토큰(sentences)": len(sentences),
    }
    for key, value in sizes.items():
        print(f"  {key:<22} {value}")

    # 2) 짝이 되어야 하는 변수끼리 개수를 비교하고, 어느 쪽이 모자란지 알려준다.
    for a, b in [("원본 문서(inputs)", "원본 정답(targets)"),
                 ("정제 문서(clean)", "정제 정답(clean_targets)"),
                 ("토큰(sentences)", "정제 정답(clean_targets)")]:
        diff = sizes[a] - sizes[b]
        if diff == 0:
            print(f"  OK  {a} = {b}")
        elif diff > 0:
            print(f"  X   {a}가 {diff}개 더 많음 → 정답이 없는 문서가 있음")
        else:
            print(f"  X   {b}가 {-diff}개 더 많음 → 문서가 없는 정답이 있음")

    # 3) 원본에서 정제를 다시 해서 "정답지"를 만든다.
    cleaned_all = [clean_text(t) for t in inputs]
    kept_idx = [i for i, c in enumerate(cleaned_all) if c]       # 남는 문서의 원본 위치
    removed_idx = [i for i, c in enumerate(cleaned_all) if not c]  # 제거되는 문서의 원본 위치
    print(f"  원본 {len(inputs)} = 남김 {len(kept_idx)} + 제거 {len(removed_idx)}")

    exp_docs = [cleaned_all[i] for i in kept_idx]
    exp_targets = [int(targets[i]) for i in kept_idx]
    cur_targets = [int(y) for y in clean_targets]

    # 4) 정답지와 현재 변수를 앞에서부터 비교해서 처음 달라지는 위치를 찾는다.
    doc_diff = next((p for p, (a, b) in enumerate(zip(exp_docs, clean)) if a != b), None)
    tgt_diff = next((p for p, (a, b) in enumerate(zip(exp_targets, cur_targets)) if a != b), None)

    if len(exp_docs) == len(clean) and doc_diff is None:
        print("  OK  정제 문서가 원본을 다시 정제한 결과와 같음")
    else:
        print(f"  X   정제 문서가 다름 (정답지 {len(exp_docs)}개 / 현재 {len(clean)}개, 처음 다른 위치 {doc_diff})")

    if len(exp_targets) == len(cur_targets) and tgt_diff is None:
        print("  OK  정제 정답이 원본에서 걸러낸 정답과 같음")
    else:
        print(f"  X   정제 정답이 다름 (정답지 {len(exp_targets)}개 / 현재 {len(cur_targets)}개, 처음 다른 위치 {tgt_diff})")

    # 5) 토큰이 정제 문서와 맞는지 일부만 확인한다. (전부 하면 오래 걸림)
    step = max(1, len(clean) // 200)
    bad = [p for p in range(0, min(len(clean), len(sentences)), step)
           if sentences[p] != word_tokenize(clean[p])]
    print("  OK  토큰이 정제 문서와 같음 (일부 확인)" if not bad else f"  X   토큰이 다른 위치: {bad[:5]}")

    # 6) 제거된 문서의 카테고리별 개수
    removed_labels = pd.Series([target_names[int(targets[i])] for i in removed_idx]).value_counts()
    display(removed_labels.to_frame("제거 수"))
    return {"kept_idx": kept_idx, "removed_idx": removed_idx}


pair_info = {
    "train": check_pairs("train", train_inputs, train_targets, train_clean, train_clean_targets, train_sentences),
    "val":   check_pairs("val",   val_inputs,   val_targets,   val_clean,   val_clean_targets,   val_sentences),
    "test":  check_pairs("test",  test_inputs,  test_targets,  test_clean,  test_clean_targets,  test_sentences),
}

> **인사이트 :** 정제 후 고유 단어 수가 감소했으며, 빈 문서는 train·val·test에서 각각 10개, 5개, 6개 추가로 발생했다. 기존 빈 문서와 정제로 추가된 빈 문서를 모두 제거한다.

| EDA 항목 | 확인한 내용 | 추가 정제 및 학습에 반영할 점 |
|---|---|---|
| 문서 길이 | 정제 후 train의 토큰 수는 중앙값 **88**, 90% 지점 **357**, 95% 지점 **607.5**, 최대 **16,002** | 1차 학습의 `max_len` 후보를 **608**로 설정. **추가 불용어 제거 후 길이 분포를 다시 확인**하여 확정 |
| 한 번 나온 토큰 | 정제 전 **90,083종(60.71%)**과 정제 후 개수·비율을 비교할 필요가 있음 | 한 번 등장했다는 이유만으로 바로 제거하지 않고, 남은 단어 예시를 확인하여 처리 여부 판단 |
| 어휘 크기 | 정제 후 고유 단어 수가 감소함 | 불용어 추가 제거 후 고유 단어 수가 얼마나 더 줄었는지 확인 |
| 빈 문서 | 정제로 train·val·test에서 각각 **10건·5건·6건** 추가 발생. 기존 빈 문서를 포함하면 각각 **316건·100건·120건**, 총 **536건** | 불용어 제거까지 마친 뒤 다시 확인하고, **빈 문서와 해당 정답 라벨을 함께 제외** |
| 분할 간 같은 본문 | 정제 후 분할 간 중복은 별도로 재확인 필요. 서로 달랐던 본문도 정제로 같아질 수 있음 | 빈 문서 제거 후에도 학습·검증·테스트에 같은 본문이 남는지 확인하고 처리 |
| 이메일·특수문자·숫자 | 이메일·특수문자 제거 및 숫자 보존 규칙을 적용함 | 잔여 이메일·특수문자가 있는지 확인. 숫자는 버전·사양·금액 등의 단서로 유지하되, 특수문자 제거로 표현이 달라진 예시도 점검 |
| 연속 토큰 쌍 | 정제 후 연속 토큰 쌍을 집계하여 정제 전 결과와 비교 | 흔한 불용어 조합이 여전히 많은지 확인하고, 불용어 제거 후 다시 비교. 정제로 새롭게 이웃한 토큰도 포함됨에 유의 |

## 공통 통일 항목
### EDA 결과로 정하는 값

`MAX_LEN`은 문서마다 모델에 넣을 토큰 개수다. 짧으면 패딩으로 채우고, 길면 뒷부분을 자른다. OOV는 사전에 없는 토큰이며 1차에서는 패딩과 같은 0번을 사용한다.

In [ ]:
# 분포 확인 후 None을 선택한 정수로 바꾼다.
MAX_LEN = 304
PAD_OOV_ID = 0
print("train 95분위 후보", candidate_max_len)

### 학습 조건

배치는 한 번에 처리할 문서 묶음이다. 학습률(`LR`)은 가중치 업데이트의 크기를 조절하며, 에폭은 train 전체를 한 바퀴 학습하는 단위다.

In [ ]:
BATCH_SIZE = 32
LR = 0.005
EPOCHS = 10

### 임베딩별로 달라지는 값

`window`는 주변 단어를 보는 범위, `min_count`는 사전에 포함할 최소 등장 횟수다. `sg=1`은 중심 단어로 주변 단어를 예측하는 Skip-gram이고, `workers=1`은 학습 스레드를 하나로 고정한다.

In [ ]:
TRAIN_EMBED_DIM = 128
GLOVE_DIM = 200
GLOVE_PATH = Path("../data/glove.6B.200d.txt")
GENSIM_OPTIONS = dict(vector_size=TRAIN_EMBED_DIM, window=5, min_count=1, sg=1, seed=SEED, workers=1)

> **인사이트 :** GloVe는 사전학습 자료와 벡터 차원이 다르다. 세 결과를 비교할 때 이 조건을 함께 기록한다.

## 벡터화

임베딩은 토큰을 숫자 벡터로 나타낸다. 사전에서 토큰의 번호를 찾으면 임베딩 행렬의 같은 번호 행에서 벡터를 꺼낼 수 있다.

In [ ]:
from gensim.models import FastText, Word2Vec

if not isinstance(MAX_LEN, int) or MAX_LEN < 1:
    raise ValueError("정제 후 train EDA를 확인하고 MAX_LEN에 양의 정수를 입력하세요.")

def make_embedding(words, vectors):
    vocab = {word: i for i, word in enumerate(words, start=1)}
    matrix = np.zeros((len(vocab) + 1, vectors.shape[1]), dtype=np.float32)
    matrix[1:] = vectors
    return vocab, matrix

Word2Vec은 train 문서에서 가까운 문맥에 등장하는 단어의 벡터를 학습한다.

In [ ]:
w2v = Word2Vec(sentences=train_sentences, **GENSIM_OPTIONS)
embeddings = {"Word2Vec": make_embedding(w2v.wv.index_to_key, w2v.wv.vectors)}
del w2v

FastText는 단어를 구성하는 글자 조각도 학습에 활용한다. 현재 입력 코드에서는 train 사전에 없는 토큰을 공통 규칙에 따라 0으로 매핑한다.

In [ ]:
fasttext = FastText(sentences=train_sentences, **GENSIM_OPTIONS)
embeddings["FastText"] = make_embedding(fasttext.wv.index_to_key, fasttext.wv.vectors)
del fasttext

GloVe는 단어가 함께 등장한 통계로 미리 학습한 벡터를 파일에서 읽는다. 파일에서 train 어휘와 겹치는 단어의 200차원 벡터를 가져온다.

In [ ]:
# Stanford GloVe 6B의 200d 텍스트 파일 경로를 지정한다.
train_vocabulary = set(clean_counts)
glove_vectors = {}
with GLOVE_PATH.open(encoding="utf-8") as glove_file:
    for line in glove_file:
        word, _, coefficients = line.partition(" ")
        if word in train_vocabulary:
            glove_vectors[word] = np.fromstring(coefficients, sep=" ", dtype=np.float32)
if not glove_vectors or any(vector.size != GLOVE_DIM for vector in glove_vectors.values()):
    raise ValueError("GloVe 파일 경로와 200차원 형식을 확인하세요.")
embeddings["GloVe"] = make_embedding(glove_vectors, np.stack(list(glove_vectors.values())))
del glove_vectors

> **인사이트 :** FastText도 공통 사전에 없는 토큰을 0번으로 처리한다. GloVe 사전은 train 어휘와 겹치는 단어만 사용한다.

## 입력 데이터 구성
### 데이터셋

Dataset은 문서 한 건을 고정 길이 정수 배열과 정답 카테고리 번호로 돌려준다. 앞의 `MAX_LEN`개 토큰을 번호로 바꾸고 남은 자리를 0으로 채운다.

In [ ]:
class TextEmbeddingDataset(Dataset):
    def __init__(self, tokens, labels, word2idx, max_len):
        self.inputs = torch.full((len(tokens), max_len), PAD_OOV_ID, dtype=torch.long)
        self.labels = torch.as_tensor(labels, dtype=torch.long)
        for i, doc in enumerate(tokens):
            ids = [word2idx.get(word, PAD_OOV_ID) for word in doc[:max_len]]
            self.inputs[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]

### 데이터 로더

DataLoader는 Dataset에서 여러 문서를 꺼내 배치로 묶는다. `shuffle`은 순서를 섞는 옵션으로 train에만 적용한다. test 로더는 최종 평가 셀에서 만들도록 되어 있다.

In [ ]:
def make_loader(tokens, labels, vocab, *, shuffle=False):
    dataset = TextEmbeddingDataset(tokens, labels, vocab, MAX_LEN)
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, generator=generator)

def make_train_val_loaders(vocab):
    return (
        make_loader(train_sentences, train_clean_targets, vocab, shuffle=True),
        make_loader(val_sentences, val_clean_targets, vocab),
    )

## 모델 구현

양방향 LSTM은 토큰을 앞에서 뒤로, 뒤에서 앞으로 각각 읽는다. 마지막 층의 두 은닉 상태(문서 요약값)를 이어 붙여 20개 클래스 점수로 바꾼다. `dropout`은 학습 중 일부 신호를 무작위로 끄고, `freeze=False`는 임베딩도 함께 업데이트한다.

In [ ]:
HIDDEN_DIM = 128
OUTPUT_DIM = len(target_names)

class EmbeddingLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, output_dim, num_layers=2, dropout=0.5):
        super().__init__()
        weights = torch.tensor(embedding_matrix, dtype=torch.float32)
        self.embedding = nn.Embedding.from_pretrained(weights, freeze=False, padding_idx=PAD_OOV_ID)
        self.lstm = nn.LSTM(
            weights.shape[1], hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout, bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        last_layer = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(last_layer)

## 학습 및 평가
### 디스코드 알림

웹훅(webhook)은 Discord 채널에 알림을 보내는 주소다. 환경변수 `DISCORD_WEBHOOK_URL`에 설정하면 모델 학습 완료 시 알림을 보낸다.

In [ ]:
import json
import urllib.request

def notify_discord(message):
    webhook = os.environ.get("DISCORD_WEBHOOK_URL")
    if not webhook:
        return
    payload = json.dumps({"content": message}).encode("utf-8")
    request = urllib.request.Request(
        webhook, data=payload, headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(request, timeout=8):
            pass
    except (OSError, ValueError) as error:
        print("Discord 알림 실패:", type(error).__name__)

### 학습 및 평가 함수

`train_epoch`는 배치마다 예측, 손실 계산, 역전파, 가중치 업데이트를 수행한다. `tqdm`은 배치 진행률을 표시한다.

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, description):
    model.train()
    loss_sum = 0.0
    for inputs, targets in tqdm(loader, desc=description, leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(inputs), targets)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(targets)
    return loss_sum / len(loader.dataset)

`evaluate`는 평가 모드에서 가중치를 고정한 채 손실과 정확도를 계산한다. `argmax`는 점수가 가장 높은 클래스 번호를 고른다.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    loss_sum = 0.0
    actual, predicted = [], []
    for inputs, targets in loader:
        logits = model(inputs.to(device))
        loss_sum += loss_fn(logits, targets.to(device)).item() * len(targets)
        actual.extend(targets.tolist())
        predicted.extend(logits.argmax(dim=1).cpu().tolist())
    return {
        "loss": loss_sum / len(loader.dataset),
        "accuracy": float(np.mean(np.equal(actual, predicted))),
        "actual": actual,
        "predicted": predicted,
    }

def report(scores, as_dict=False):
    return classification_report(
        scores["actual"], scores["predicted"], labels=list(range(OUTPUT_DIM)),
        target_names=target_names, digits=3, zero_division=0, output_dict=as_dict,
    )

### 베이스라인 3종 학습

Adam은 기울기를 이용해 가중치를 업데이트하는 옵티마이저다. `CrossEntropyLoss`는 클래스 점수와 정답 번호를 비교한다. 검증 손실이 가장 낮을 때의 가중치 파일을 체크포인트로 저장한다.

In [ ]:
def fit_baseline(name, vocab, matrix):
    seed_everything(SEED)
    model = EmbeddingLSTM(matrix, HIDDEN_DIM, OUTPUT_DIM).to(device)
    train_loader, val_loader = make_train_val_loaders(vocab)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    checkpoint_dir = Path("checkpoints")
    checkpoint_dir.mkdir(exist_ok=True)
    checkpoint = checkpoint_dir / f"baseline_{name.lower()}.pt"
    history = []
    best_loss = float("inf")
    best_epoch = None
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, f"{name} {epoch}/{EPOCHS}")
        scores = evaluate(model, val_loader, loss_fn)
        history.append(dict(epoch=epoch, train_loss=train_loss,
                            val_loss=scores["loss"], val_accuracy=scores["accuracy"]))
        print(name, epoch, f"train={train_loss:.4f} val={scores['loss']:.4f} acc={scores['accuracy']:.3f}")
        if scores["loss"] < best_loss:
            best_loss, best_epoch = scores["loss"], epoch
            torch.save(model.state_dict(), checkpoint)
    if best_epoch is None:
        raise RuntimeError(f"{name}: 검증 손실이 유한한 에폭이 없습니다.")
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    scores = evaluate(model, val_loader, loss_fn)
    notify_discord(f"{name} 학습 완료 | 최적 에폭 {best_epoch} | val 정확도 {scores['accuracy']:.3f}")
    return dict(history=pd.DataFrame(history), best_epoch=best_epoch,
                scores=scores, checkpoint=checkpoint)

runs = {}
for name, (vocab, matrix) in embeddings.items():
    runs[name] = fit_baseline(name, vocab, matrix)

### 손실 곡선 비교

학습 손실과 검증 손실을 임베딩별로 비교한다. train 손실은 계속 줄고 val 손실이 올라가는 시점이 있으면 과적합 가능성을 살펴본다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for axis, metric in zip(axes, ("train_loss", "val_loss")):
    for name, run in runs.items():
        history = run["history"]
        axis.plot(history["epoch"], history[metric], marker="o", label=name)
    axis.set(xlabel="Epoch", ylabel="Cross-entropy loss", title=metric)
    axis.legend()
plt.tight_layout()
plt.show()

> **인사이트 :** 세 임베딩 모두 train loss는 지속해서 감소했으나, val loss는 최저점 이후 상승하여 과적합 징후를 보였다. 임베딩별로 과적합이 나타나는 시점이 달랐으며, 비교에는 각 모델의 val loss가 가장 낮은 에폭의 가중치를 사용한다.

### 검증 분류 리포트

Precision은 해당 클래스로 예측한 문서 중 맞힌 비율이고, recall은 실제 그 클래스 문서 중 맞힌 비율이다. F1은 둘의 조화평균이다. Macro 평균은 모든 클래스를 같은 비중으로, weighted 평균은 클래스별 문서 수에 비례해 반영한다.

In [ ]:
for name, run in runs.items():
    print(name, "최소 val loss 에폭", run["best_epoch"])
    print(report(run["scores"]))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 각 모델의 최소 val loss 에폭에서 얻은 검증 예측으로 그린다.
for name, run in runs.items():
    cm = confusion_matrix(
        run["scores"]["actual"],       # 실제 정답
        run["scores"]["predicted"],    # 모델의 예측
        labels=list(range(len(target_names))),
    )

    fig, ax = plt.subplots(figsize=(15, 13))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=target_names,
    ).plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=90,
        values_format="d",
        colorbar=True,
    )

    ax.set_title(
        f"{name} | Validation | Best epoch: {run['best_epoch']}"
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    plt.tight_layout()
    plt.show()

## 계획서 결과 기록
### 베이스라인 3종 비교 (val 기준)

> 조건: 시드 42, 배치 32, max_len ○○, 에폭 10, lr 0.005 (3종 동일)

| 임베딩 | 차원 | 사전 크기 | OOV 비율 | 마지막 train loss | val 정확도 | 최소 val loss | 최고 val 정확도 | 최적 에폭 | macro F1 (EDA 이후 선택) | 메모 |
|---|---|---|---|---|---|---|---|---|---|---|
| Word2Vec | 128 | 89,259 | 0.0180 | 0.1222 | 0.6102 | 1.3879 | 0.6353 | 5 | 0.5981 | |
| FastText | 128 | 89,259 | 0.0180 | 0.1594 | 0.6187 | 1.4005 | 0.6500 | 6 | 0.6059 | |
| GloVe | 200 | 47,334 | 0.5085 | 0.3263 | 0.6018 | 1.3484 | 0.6263 | 7 | 0.5931 | |


OOV 비율은 길이 제한을 적용한 뒤 검증 토큰 중 각 임베딩 사전에 없는 토큰의 비율이다. 표의 최고 val 정확도는 전체 에폭에서 구하고, val 정확도와 macro F1은 최소 val loss 에폭의 모델에서 구한다.

In [ ]:
def oov_rate(tokens, vocab):
    model_tokens = (word for doc in tokens for word in doc[:MAX_LEN])
    total, missing = 0, 0
    for word in model_tokens:
        total += 1
        missing += word not in vocab
    return missing / total if total else float("nan")

print(f"시드 {SEED} / 배치 {BATCH_SIZE} / max_len {MAX_LEN} / 에폭 {EPOCHS} / lr {LR}")
rows = []
for name, run in runs.items():
    vocab, matrix = embeddings[name]
    history, scores = run["history"], run["scores"]
    rows.append({
        "임베딩": name, "차원": matrix.shape[1], "사전 크기": len(vocab),
        "val OOV 비율": oov_rate(val_sentences, vocab),
        "마지막 train loss": history["train_loss"].iloc[-1],
        "val 정확도": scores["accuracy"], "최소 val loss": scores["loss"],
        "최고 val 정확도": history["val_accuracy"].max(), "최적 에폭": run["best_epoch"],
        "macro F1": report(scores, as_dict=True)["macro avg"]["f1-score"], "메모": "",
    })
baseline_results = pd.DataFrame(rows)
display(baseline_results.round(4))

| 비교할 점 | 이번 결과 해석 |
|---|---|
| **선택된 모델의 정확도** | FastText가 **61.87%** 로 가장 높음 |
| **카테고리를 같은 비중으로 평가한 macro F1** | FastText가 **0.6059**로 가장 높음 |
| **최소 검증 손실** | GloVe가 **1.3484**로 가장 낮음 |
| **모르는 토큰 비율** | Word2Vec·FastText는 **1.80%**, GloVe는 **50.85%** 로 차이가 큼 |
| **과적합** | Word2Vec의 마지막 train loss가 가장 낮지만, 앞서 본 val loss 상승을 고려하면 이것만으로 가장 좋다고 볼 수 없음 |

> **인사이트 :** 수치상으로는 FastText가 가장 좋은 수치를 기록하여 최종 모델로 선정한다.

## 최종 선발 및 최종 평가

val 결과와 실험 기록을 보고 모델을 확정한 다음, 이 코드의 주석을 직접 풀어 test를 한 번 평가한다.

In [ ]:
# 검증과 실험을 마친 뒤 선택한 모델명으로 변경한다.
FINAL_MODEL = "FastText"
if FINAL_MODEL is not None and "final_test_scores" not in globals():
    final_vocab, final_matrix = embeddings[FINAL_MODEL]
    final_model = EmbeddingLSTM(final_matrix, HIDDEN_DIM, OUTPUT_DIM).to(device)
    final_model.load_state_dict(torch.load(
        runs[FINAL_MODEL]["checkpoint"], map_location=device, weights_only=True
    ))
    test_loader = make_loader(test_sentences, test_clean_targets, final_vocab)
    final_test_scores = evaluate(final_model, test_loader, nn.CrossEntropyLoss())
    print(FINAL_MODEL, "test 정확도", final_test_scores["accuracy"])
    print(report(final_test_scores))

# 마치며

| 목차 | 작성 내용 |
| :--- | :--- |
| **1. 실험 개요 (Summary)** | |
| **◦ 실험명 및 기간**<br>언제부터 언제까지 진행했는지 명시합니다. | • **실험명**: 뉴스의 카테고리를 예측하는 딥러닝 모델을 구현<br>• **기간**: 2026년 11월 11일 → 2026년 11월 17일 |
| **◦ 핵심 가설 (Hypothesis)**<br>"만약 [A]를 하면, [B] 때문에 [C 지표]가 [N%] 개선될 것이다" 형태로 적습니다. | 가설 없이 실시 |
| **2. 실험 결과 (Results)** | |
| **◦ 정량적 데이터**<br>목표했던 핵심 지표(KPI)와 보조 지표의 변화를 숫자로 명확히 보여줍니다. | Word2Vec, FastText, GloVe 방식으로 입력 데이터를 벡터화하여 임베딩 행렬을 생성하고, 평가 지표(Accuracy, Precision, Recall, F1-score 등)를 활용해 모델 성능을 분석하고 비교하는 미션이었습니다.<br><br>• **비교 기준**: 임베딩, 차원, 사전 크기, OOV 비율, 마지막 train loss, val 정확도, 최소 val loss, 최고 val 정확도, 최적 에폭, macro F1<br>• **성능 비교**: `val 정확도`를 기준으로 **FastText가 61.87%**로 가장 높았으며, `macro F1` 또한 **0.6059**로 가장 높았습니다. 최소 val loss는 GloVe가 1.3484로 가장 낮았으나, 모르는 토큰 비율(OOV)이 **50.85%**로 높아 최종 모델에서 제외했습니다.<br>• **최종 결과**: Word2Vec과 FastText의 성능이 유사하나 val 정확도와 macro F1을 기준으로 최고 성능을 보인 **FastText를 최종 모델로 선정**했으며, **test 정확도 0.6312**를 확인했습니다. |
| **◦ 가설 검증 여부**<br>가설이 입증(Win)되었는지, 기각(Lose)되었는지, 혹은 유의미한 차이가 없었는지(Inconclusive) 결론을 내립니다. | **1. 클래스 불균형 대응 한계**<br>클래스 분포가 불균형하다고 파악하여 비율로도 분류했으나, test에서 데이터가 가장 적은 `talk.religion.misc` 카테고리의 성능이 낮게 나왔습니다. 비율 조정만으로는 소수 클래스 학습에 한계가 있다고 판단했습니다.<br>• `talk.religion.misc`: Precision `0.218` / Recall `0.197` / F1-score `0.207` / Support `122`<br>• `weighted avg`: Precision `0.646` / Recall `0.631` / F1-score `0.633`<br><br>**2. 데이터 정제 및 문서 길이(`max_len`) 변화**<br>정제 전 EDA에서 `max_len`은 **705**였으나, 정제 이후 **304**로 줄어들었습니다. 본문의 이메일, 특수문자 제거 및 영어 소문자 변환 과정에서 빈 문서가 train 기준 306건에서 316건으로 10건 증가(val 5건, test 6건 추가)했고, 빈 문서를 전부 제거하면서 `max_len` 설정치가 감소하는 것을 확인했습니다.<br><br>**3. N-gram 토큰 변화**<br>정제 전 n-gram에서는 특수문자(`('>', "'") : 49,913`)가 가장 많았으나, 정제 이후 `('ax', 'ax') : 41,675`로 변경되었습니다. 특수문자는 제거되었으나, 정제 후에도 의미 없는 반복 토큰이 상위에 남아 추가 확인이 필요했습니다. |
| **3. 원인 분석 (Why & Insights)** | |
| **◦ 가설 검증 및 관찰 결과**<br>왜 이런 결과가 나왔는지 데이터를 바탕으로 분석합니다. | **• 성공 요인**<br>이번 미션은 베이스라인 3종 비교를 중심으로 진행했으며, 정제 효과와 하이퍼파라미터 변경 효과를 검증하는 추가 실험은 수행하지 않았습니다. 평가 지표의 구체적인 사용법은 강의에서 다루지 않았으나, 기존 머신러닝에서 배운 내용을 토대로 LSTM 기반 텍스트 분류에 성공적으로 적용한 점이 성과입니다.<br><br>**• 실패 요인**<br>1. **추가 실험 미완료**: 하이퍼파라미터(`BATCH_SIZE = 32`, `LR = 0.005`, `EPOCHS = 10`) 변경 실험과 NLTK 외 다른 토크나이저 적용 실험까지 계획했으나, 시간 부족으로 미완료 상태로 남았습니다.<br>2. **EDA 해석 부족**: 현업에서 사용되는 워드클라우드(WordCloud)를 시도했으나, 활용법을 충분히 익히지 못해 결과에 대한 해석을 도출하지 못한 점이 아쉽습니다. |
| **◦ 예상치 못한 발견**<br>목표 지표 외에 새롭게 알게 된 인사이트를 기록합니다. | 이번 미션을 통해 **정제 전, 정제 후, 그리고 실험 설계를 위해 각각 필요한 EDA가 다르다**는 점을 깨달았습니다. 기존에는 단순히 모델 학습 전후의 EDA만 진행했다면, 이번에는 정제 전후 비교부터 실험 계획 수립을 위한 EDA까지 연결하며 한 단계 성장했음을 느꼈습니다.<br><br>앞으로도 이러한 인사이트와 회고록을 수치 기반으로 기록하는 연습을 지속하여, 향후 다른 미션에서도 같은 템플릿을 활용해 성장 과정을 비교해 나갈 계획입니다. |
| **4. 실험 과정 회고 (Process Review)** | |
| **◦ 잘한 점 (Keep)**<br>실험 설계나 실행 과정에서 효율적이었던 부분을 적습니다. | • **구체적인 계획서 작성**: 목차부터 사용 도구, 의사코드, 명령어까지 지정하여 다른 사람도 동일한 실험 구성을 재현할 수 있도록 문서화한 점<br>• **과감한 결단**: 계획했던 추가 실험을 진행하지 못한 아쉬움은 있지만, 마감 기한 내 핵심 미션 완료를 우선순위로 두고 과감히 선택한 점<br>• **이슈 및 작업 일지 정리**: 여러 요일에 걸쳐 작업하면서 그날 발생한 이슈들을 사전에 정리해 두어 언제든 맥락을 파악할 수 있도록 문서화한 점 |
| **◦ 아쉬운 점 (Problem)**<br>실험 자체의 한계점이나 실수, 개선 방향을 기록합니다. | • **시간 관리 및 실험 미실시**: 태스크를 구체화하고 계획을 세우는 데 시간이 길어지면서 수행 시간이 부족해졌고, 계획했던 추가 실험을 실행하지 못한 점<br>• **러닝 커브**: 일부 EDA 기법에 대한 사전 학습이 부족해 인사이트를 뽑아내지 못한 점<br>• **데이터 전처리 및 코드 관리 실수**<br>  - 빈 문서를 제거할 때 해당 라벨도 함께 제외해야 한다는 점<br>  - 노트북 셀 실행 순서와 변수 재할당이 데이터 불일치를 유발할 수 있다는 점<br>  - 단순 개수뿐만 아니라 문서와 토큰의 실제 내용 및 순서까지 교차 확인해야 한다는 점<br>• **향후 개선 방향**: 다음 실험부터는 코드를 작성하기 전에 변수의 역할과 데이터 처리 순서를 먼저 명확히 정리하고 진행할 것 |
